# Conv1 FPGA verification and timing

Run all cells on the PYNQ-Z2. Keep `conv1.bit`, its matching `conv1.hwh`, and the project's `pynq_conv1.py` beside this notebook. Put the five exported `.mem` files in the same directory or in `Mixed_precision_firstlayer/`. Each case uses its own CPU reference and must match all 16,384 outputs. This overlay runs ResNet-20 Conv1 with INT8 activations and INT4 weights.


In [ ]:
from pathlib import Path
from time import perf_counter

from pynq import Clocks
from pynq_conv1 import PynqConv1, read_mem

root = Path.cwd()
bitstream = root / "conv1.bit"
export_dir = root if (root / "expected.mem").is_file() else root / "Mixed_precision_firstlayer"
# Add another exported Conv1 vector directory here to test another image.
cases = {"Exported Conv1": export_dir}

for path in (bitstream, bitstream.with_suffix(".hwh")):
    if not path.is_file():
        raise FileNotFoundError(path)

# Validate every file before programming the board.
sizes = {"input": 3072, "weights": 54, "bias": 16,
         "multiplier": 16, "expected": 16384}
vectors = {
    name: {kind: read_mem(directory / (kind + ".mem"), length)
           for kind, length in sizes.items()}
    for name, directory in cases.items()
}


In [ ]:
start = perf_counter()
fpga = PynqConv1(bitstream)
program_ms = (perf_counter() - start) * 1000
clock_mhz = Clocks.fclk0_mhz
print(f"FPGA setup and programming: {program_ms:.2f} ms")
print(f"PL clock: {clock_mhz:.3f} MHz (the supplied overlay targets 50 MHz)")


In [ ]:
def verify_case(name, data):
    start = perf_counter()
    for op, kind in ((1, "input"), (2, "weights"), (3, "bias")):
        for index, word in enumerate(data[kind]):
            fpga.command(op, index, word)
    for index, word in enumerate(data["multiplier"]):
        fpga.command(4, index, word)
        fpga.command(5, index, word >> 32)

    loaded = perf_counter()
    fpga.command(6)
    deadline = loaded + 5
    # Status bits [2:1] are done and busy. The START command waits for ACK.
    while (fpga.replies.read(8) & 0b110) != 0b100:
        if perf_counter() >= deadline:
            raise TimeoutError(f"{name}: PL did not finish Conv1")

    done = perf_counter()
    actual = [fpga.command(7, index) for index in range(sizes["expected"])]
    read_back = perf_counter()
    cycles, image_reads, packed_reads, writes = (
        fpga.command(op) for op in range(8, 12)
    )
    mismatches = [
        index for index, (want, got) in enumerate(zip(data["expected"], actual))
        if want != got
    ]

    print(f"{name}: {len(actual) - len(mismatches)}/{len(actual)} outputs match")
    print(f"  FPGA compute: {cycles:,} cycles / {clock_mhz:.3f} MHz "
          f"= {cycles / clock_mhz / 1000:.3f} ms")
    print(f"  ARM load: {(loaded - start) * 1000:.3f} ms")
    print(f"  ARM START to observed done: {(done - loaded) * 1000:.3f} ms")
    print(f"  ARM output readback: {(read_back - done) * 1000:.3f} ms")
    print(f"  ARM load through readback: {(read_back - start) * 1000:.3f} ms")
    print(f"  Counters: {image_reads:,} image reads, {packed_reads:,} packed-word reads, "
          f"{writes:,} output writes")
    if mismatches:
        index = mismatches[0]
        channel, pixel = divmod(index, 32 * 32)
        row, col = divmod(pixel, 32)
        print(f"  First mismatch at CHW ({channel}, {row}, {col}): "
              f"expected {data['expected'][index]}, got {actual[index]}")
    return not mismatches and writes == sizes["expected"] and cycles > 0


results = {}
for name, data in vectors.items():
    results[name] = verify_case(name, data)
    print(f"  {'PASS' if results[name] else 'FAIL'}\n")

if not all(results.values()):
    raise AssertionError("Conv1 failed: " + ", ".join(name for name in results if not results[name]))
print(f"PASS: {len(results)}/{len(vectors)} cases")


The FPGA compute time comes from the Conv1 cycle counter divided by the PL clock frequency. START to observed done includes Python and MMIO polling. Load through readback also includes the input, parameter, and output transfers. The software comparison, counter queries, and one-time FPGA programming are outside that total.
